In [1]:
import pandas as pd

df_answers = pd.read_csv("../data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [2]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(
        description="Reasoning about the quality of the answer."
    )
    score: Literal["good", "bad"] = Field(
        description="'good' if the answer is correct and complete, 'bad' otherwise."
    )

In [3]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

In [4]:
aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [6]:
from dotenv import load_dotenv
from openai import OpenAI
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry, map_progress

load_dotenv()
openai_client = OpenAI()

In [8]:
import pandas as pd

df_answers = pd.read_csv("../data/rag-answers-new.csv")
answers = df_answers.to_dict(orient="records")

In [9]:
rec = answers[0]
rec

{'question': 'Is it okay to join the course late if I just found it now?',
 'answer_llm': 'Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [10]:
prompt = aqa_judge_prompt.format(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)
print(prompt)

Question:
Is it okay to join the course late if I just found it now?

Original Answer (ground truth):
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.

AI Answer:
Yes, you can still join the course late. If you want a certificate, though, you need to submit your project while submissions are still being accepted.


In [11]:
eval_result, usage = llm_structured_retry(
    openai_client,
    aqa_judge_instructions,
    prompt,
    AnswerEvaluation,
)

eval_result

AnswerEvaluation(reasoning='The AI answer preserves the full meaning of the ground truth: late joining is allowed, and certificate eligibility depends on submitting the project before submissions close. It is semantically equivalent.', score='good')

In [12]:
calc_price(usage)

{'input_cost': 0.00022275000000000002,
 'output_cost': 0.0002295,
 'total_cost': 0.00045225}

In [13]:
def evaluate_aqa(question, answer_orig, answer_llm, model="gpt-5.4-mini"):
    prompt = aqa_judge_prompt.format(
        question=question,
        answer_orig=answer_orig,
        answer_llm=answer_llm
    )

    result, usage = llm_structured_retry(
        openai_client,
        aqa_judge_instructions,
        prompt,
        AnswerEvaluation,
        model=model,
    )

    return result, usage

In [14]:
eval_result, usage = evaluate_aqa(
    question=rec["question"],
    answer_orig=rec["answer_orig"],
    answer_llm=rec["answer_llm"]
)

eval_result

AnswerEvaluation(reasoning='The AI answer conveys the same meaning as the ground truth: late enrollment is okay, but certificate eligibility depends on submitting the project while submissions are still open. No key information is missing or incorrect.', score='good')

In [15]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(
        question=rec["question"],
        answer_orig=rec["answer_orig"],
        answer_llm=rec["answer_llm"]
    )

    result = {
        "question": rec["question"],
        "document": rec["document"],
        "score": eval_result.score,
        "reasoning": eval_result.reasoning,
    }

    return result, usage

In [16]:
from concurrent.futures import ThreadPoolExecutor

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, answers, judge_record)

  0%|          | 0/395 [00:00<?, ?it/s]

In [17]:
results[10]

({'question': 'How do I join the Office Hours or live workshop if I don’t have the Zoom link?',
  'document': '489dd1c9d9',
  'score': 'good',
  'reasoning': 'The AI answer matches the ground truth: it says students do not need the Zoom link, can join via YouTube Live, find the URL in Telegram/Slack announcements, watch on the DataTalksClub YouTube channel, and ask questions via Slido. It preserves the key point that Zoom is for instructors/presenters/TAs only, though that detail is omitted; the omission does not change the answer for students.'},
 ResponseUsage(input_tokens=433, input_tokens_details=InputTokensDetails(cached_tokens=0, cache_write_tokens=0), output_tokens=102, output_tokens_details=OutputTokensDetails(reasoning_tokens=0), total_tokens=535))

In [18]:
evaluations = []
usages = []

for evaluation, usage in results:
    evaluations.append(evaluation)
    usages.append(usage)

In [19]:
calc_total_price(usages)

0.25359450000000006

In [20]:
df_eval = pd.DataFrame(evaluations)

In [21]:
df_eval.head()

,question,document,score,reasoning
0,Is it okay to join the course late if I just f...,74eb249bbf,good,The AI answer preserves the key meaning of the...
1,Can I still take this course even if I missed ...,74eb249bbf,good,The AI answer preserves the key meaning of the...
2,If I join after the course has already started...,74eb249bbf,good,The AI answer preserves the core meaning: join...
3,Do I need to submit my project before submissi...,74eb249bbf,good,The AI answer preserves the ground truth meani...
4,I’m a bit late to the course—what do I need to...,74eb249bbf,good,The AI answer preserves the key condition from...


In [22]:
df_eval.score.value_counts()

score
good    379
bad      16
Name: count, dtype: int64

In [23]:
df_eval.score.value_counts(normalize=True)

score
good    0.959494
bad     0.040506
Name: proportion, dtype: float64

In [24]:
df_eval[df_eval["score"] == "bad"].head()

,question,document,score,reasoning
15,How do the free GPU hours work on these cloud ...,c6c2888275,bad,The AI answer does not address the question at...
29,Is peer-review of the capstone project require...,69d122f12e,bad,The AI answer contradicts the ground truth. Th...
38,How will I know when a module is actually read...,96286b4be4,bad,The AI answer does not convey the ground truth...
106,Do I need an OpenAI API key just to check how ...,fe8fed31e6,bad,The ground truth says no API key is needed and...
124,What model or provider should I switch to if I...,6e1d8a7b29,bad,The ground truth gives specific alternatives t...


In [25]:
df_eval.to_csv("../data/rag-evaluations-new.csv", index=False)
